# MCP

模型上下文协议 (MCP)是一个开放协议，它规范了应用程序向语言模型提供工具和上下文的方式。LangGraph 代理可以通过该langchain-mcp-adapters库使用 MCP 服务器上定义的工具。

<img src="https://langchain-ai.github.io/langgraph/agents/assets/mcp.png">


## 向 MCP 服务器进行身份验证¶
您可以设置自定义身份验证中间件，通过 MCP 服务器对用户进行身份验证，从而访问 LangGraph 平台部署中用户范围的工具。

### 此流程的示例架构：
此流程的示例架构：
<img src="https://cdn.mathpix.com/snip/images/2DUBoScbXJv4txjQkzuKU4NVk2ijKryEVk6032Ds89E.original.fullsize.png" />

## 使用 MCP 工具¶
该`langchain-mcp-adapters`软件包使代理能够使用在一个或多个 MCP 服务器上定义的工具。

### 使用 MCP 服务器上定义的工具的代理

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

client = MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            # Replace with absolute path to your math_server.py file
            "args": ["/path/to/math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()
agent = create_react_agent(
    "anthropic:claude-3-7-sonnet-latest",
    tools
)
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)

### 使用带有 ToolNode 的 MCP 工具的工作流程

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# Initialize the model
model = init_chat_model("anthropic:claude-3-5-sonnet-latest")

# Set up MCP client
client = MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            # Make sure to update to the full absolute path to your math_server.py file
            "args": ["./examples/math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # make sure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp/",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

# Bind tools to model
model_with_tools = model.bind_tools(tools)

# Create ToolNode
tool_node = ToolNode(tools)

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

# Define call_model function
async def call_model(state: MessagesState):
    messages = state["messages"]
    response = await model_with_tools.ainvoke(messages)
    return {"messages": [response]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_model")
builder.add_conditional_edges(
    "call_model",
    should_continue,
)
builder.add_edge("tools", "call_model")

# Compile the graph
graph = builder.compile()

# Test the graph
math_response = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)

## 自定义 MCP 服务器¶
要创建您自己的 MCP 服务器，您可以使用mcp库。该库提供了一种简单的方法来定义工具并将其作为服务器运行。

安装 MCP 库：

`pip install mcp`

使用以下参考实现来通过 MCP 工具服务器测试您的代理。

In [ ]:
## 数学服务器示例（stdio 传输）
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
# 天气服务器示例（可流式传输的 HTTP 传输）
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")


## 其他资源¶
- [MCP 文档](https://modelcontextprotocol.io/introduction)
- [MCP 传输文档](https://modelcontextprotocol.io/docs/concepts/transports)
- [langchain_mcp_adapters](https://github.com/langchain-ai/langchain-mcp-adapters)

# LangGraph 服务器中的 MCP 端点

模型上下文协议 (MCP)是一种开放协议，用于以与模型无关的格式描述工具和数据源，使 LLM 能够通过结构化 API 发现和使用它们。

LangGraph 服务器使用`Streamable HTTP` 传输实现 `MCP` 。这使得 LangGraph代理可以作为MCP 工具公开，从而可与任何支持 Streamable HTTP 的 MCP 兼容客户端一起使用。

MCP 端点可在`LangGraph Server/mcp`上找到。

## 要求¶
要使用 MCP，请确保已安装以下依赖项：

- langgraph-api >= 0.2.3
- langgraph-sdk >= 0.1.61

使用以下命令安装它们：

`pip install "langgraph-api>=0.2.3" "langgraph-sdk>=0.1.61"`

### 将代理公开为 MCP 工具¶
部署后，您的代理将作为工具出现在 MCP 端点中，并具有以下配置：

- 工具名称：代理的名称。
- 工具描述：代理的描述。
- 工具输入模式：代理的输入模式。
### 设置名称和描述¶
您可以在以下位置`langgraph.json`设置代理的名称和描述：

```json
{
  "graphs": {
    "my_agent": {
      "path": "./my_agent/agent.py:graph",
      "description": "A description of what the agent does"
    }
  },
  "env": ".env"
}
```

部署后，您可以使用 LangGraph SDK 更新名称和描述。

### 架构¶
定义清晰、最小的输入和输出模式，以避免向 LLM 暴露不必要的内部复杂性。

默认的MessagesState使用AnyMessage，它支持多种消息类型，但对于直接 LLM 公开来说太通用了。

相反，定义使用明确类型的输入和输出结构的自定义代理或工作流。

例如，回答文档问题的工作流程可能如下所示：

API 参考：StateGraph |开始|结束

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Define input schema
class InputState(TypedDict):
    question: str

# Define output schema
class OutputState(TypedDict):
    answer: str

# Combine input and output
class OverallState(InputState, OutputState):
    pass

# Define the processing node
def answer_node(state: InputState):
    # Replace with actual logic and do something useful
    return {"answer": "bye", "question": state["question"]}

# Build the graph with explicit schemas
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(answer_node)
builder.add_edge(START, "answer_node")
builder.add_edge("answer_node", END)
graph = builder.compile()

# Run the graph
print(graph.invoke({"question": "hi"}))

有关更多详细信息，请参阅低级概念指南。

## 使用概述¶
要启用 MCP：

升级到使用 langgraph-api>=0.2.3 版本。如果您正在部署 LangGraph 平台，则在创建新修订版本时将自动完成此操作。

MCP 工具（代理）将自动公开。

与任何支持 Streamable HTTP 的 MCP 兼容客户端连接。

### 客户¶
使用兼容 MCP 的客户端连接到 LangGraph 服务器。以下示例展示了如何使用langchain-mcp-adapters进行连接。

使用以下方式安装适配器：

`pip install langchain-mcp-adapters`

以下是如何连接到远程 MCP 端点并使用代理作为工具的示例：

API 参考：load_mcp_tools | create_react_agent

In [ ]:
# Create server parameters for stdio connection
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import asyncio

from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

server_params = {
    "url": "https://mcp-finance-agent.xxx.us.langgraph.app/mcp",
    "headers": {
        "X-Api-Key":"lsv2_pt_your_api_key"
    }
}

async def main():
    async with streamablehttp_client(**server_params) as (read, write, _):
        async with ClientSession(read, write) as session:
            # Initialize the connection
            await session.initialize()

            # Load the remote graph as if it was a tool
            tools = await load_mcp_tools(session)

            # Create and run a react agent with the tools
            agent = create_react_agent("openai:gpt-4.1", tools)

            # Invoke the agent with a message
            agent_response = await agent.ainvoke({"messages": "What can the finance agent do for me?"})
            print(agent_response)

if __name__ == "__main__":
    asyncio.run(main())

## 会话行为¶
当前的 LangGraph MCP 实现不支持会话。每个/mcp请求都是无状态且独立的。

## 验证¶
该/mcp端点使用与 LangGraph API 其他部分相同的身份验证。有关设置详情，请参阅身份验证指南。

## 禁用 MCP¶
要禁用 MCP 端点，请在配置文件中设置disable_mcp为：truelanggraph.json

```json
{
  "http": {
    "disable_mcp": true
  }
}
```

这将防止服务器暴露/mcp端点。